# Signature & Cachet PDF — Détection GPT-4o Vision

Ce notebook teste l'approche **GPT-4o vision** pour détecter les zones de signature sur des PDFs scannés.

## Principe
- Toutes les pages sont converties en images
- **1 seul appel API** GPT-4o avec toutes les images
- GPT-4o retourne un JSON avec les zones détectées (coordonnées en % de la page)
- On convertit % → points PDF et on place signature + cachet

## Quand l'utiliser ?
- PDF scanné : pas de couche texte → pymupdf ne peut pas détecter les zones
- En production : cette approche est le **fallback** quand pymupdf ne trouve rien

## Structure attendue
```
rd/signature/
  assets/
    signature.jpg
    cachet.png
  input/
    document.pdf    ← idéalement un PDF scanné pour tester
  output/
    signé_gpt4o.pdf ← résultat généré ici
```

In [ ]:
!pip install pymupdf openai pillow matplotlib -q

## 0. Configuration

In [ ]:
import os
import json
import base64
import fitz
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from openai import OpenAI

BASE       = Path(".")
PDF_INPUT  = BASE / "input" / "document.pdf"   # ← change ici
PDF_OUTPUT = BASE / "output" / "signé_gpt4o.pdf"
SIGNATURE  = BASE / "assets" / "signature.jpg"
CACHET     = BASE / "assets" / "cachet.png"

# Lieu et date à insérer dans 'Fait à …, le …'
LIEU     = "Casablanca"
DATE_STR = "16/03/2026"

# Dimensions des images en points PDF
SIG_W, SIG_H   = 120, 50
CAC_W, CAC_H   = 100, 100
SIG_MX, SIG_MY = 50, 40   # marges fallback signature (depuis bord droit/bas)
CAC_MX, CAC_MY = 50, 40   # marges fallback cachet    (depuis bord gauche/bas)

# Clé OpenAI — renseigner ici ou via variable d'environnement OPENAI_API_KEY
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "sk-...")

# Zoom pour la conversion page → image (1.5 = bonne résolution, raisonnable en tokens)
ZOOM = 1.5

for d in [BASE/"input", BASE/"output", BASE/"assets"]:
    d.mkdir(parents=True, exist_ok=True)

print("PDF input  :", PDF_INPUT.exists(), f"({PDF_INPUT})")
print("Signature  :", SIGNATURE.exists())
print("Cachet     :", CACHET.exists())
print("API Key    :", "configurée" if OPENAI_API_KEY and not OPENAI_API_KEY.startswith("sk-...") else "⚠ à renseigner")

## 1. Conversion des pages en images base64

Chaque page est rendue en PNG puis encodée en base64 pour l'API OpenAI.

In [ ]:
def pages_to_base64(doc: fitz.Document, zoom: float = 1.5) -> list[dict]:
    """
    Convertit toutes les pages d'un PDF en images base64.

    Input  : doc  — fitz.Document ouvert
             zoom — facteur d'agrandissement (1.5 = bon compromis qualité/tokens)
    Output : list de dicts {page_num, b64, width_pt, height_pt}
             width_pt / height_pt = dimensions originales en points PDF (pour la conversion %→pt)
    """
    pages = []
    for i, page in enumerate(doc):
        pix    = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        b64    = base64.b64encode(pix.tobytes("png")).decode("utf-8")
        pages.append({
            "page_num":  i + 1,
            "b64":       b64,
            "width_pt":  page.rect.width,
            "height_pt": page.rect.height,
        })
    return pages


doc = fitz.open(str(PDF_INPUT))
pages_data = pages_to_base64(doc, ZOOM)
doc.close()

print(f"PDF : {len(pages_data)} page(s) converties en images")
for p in pages_data:
    size_kb = len(p["b64"]) * 3 / 4 / 1024
    print(f"  Page {p['page_num']:3d} : {p['width_pt']:.0f}×{p['height_pt']:.0f}pt — image {size_kb:.0f} Ko")

## 2. Appel GPT-4o Vision — 1 seul appel pour toutes les pages

On envoie toutes les images en une seule requête.
GPT-4o retourne un JSON avec les zones détectées par page.

**Format de réponse attendu :**
```json
{
  "pages": [
    {
      "page": 1,
      "fait_a":    {"x_pct": 0.10, "y_pct": 0.87},
      "signature": {"x_pct": 0.55, "y_pct": 0.92},
      "cachet":    null
    },
    ...
  ]
}
```
- `x_pct` / `y_pct` = fraction de la largeur/hauteur de page (0.0 = bord gauche/haut, 1.0 = bord droit/bas)
- `y_pct` = **bord bas du label** → l'image sera placée juste en dessous
- `null` = zone non trouvée sur cette page

In [ ]:
def detect_zones_gpt4o(pages_data: list[dict], api_key: str) -> list[dict]:
    """
    Envoie toutes les pages à GPT-4o vision en un seul appel.
    Retourne la liste des zones détectées par page.

    Input  : pages_data — liste de dicts issus de pages_to_base64()
             api_key    — clé OpenAI
    Output : list de dicts par page :
             [
               {
                 "page":      int,
                 "fait_a":    {"x_pct": float, "y_pct": float} | None,
                 "signature": {"x_pct": float, "y_pct": float} | None,
                 "cachet":    {"x_pct": float, "y_pct": float} | None,
               },
               ...
             ]
    """
    client = OpenAI(api_key=api_key)
    n = len(pages_data)

    # ── Prompt système ─────────────────────────────────────────
    system_prompt = """Tu analyses des pages de documents PDF marocains (appels d'offres, cahiers des charges).
Tu dois détecter les zones de signature sur chaque page.
Réponds UNIQUEMENT avec un JSON valide, sans markdown, sans explication."""

    # ── Prompt utilisateur ────────────────────────────────────
    user_text = f"""Voici {n} pages d'un document PDF (dans l'ordre, page 1 à {n}).

Pour chaque page, détecte si présent :
1. "fait_a" : la ligne contenant "Fait à" ou "Fait a" suivi de points/tirets/blancs et "le" avec un espace pour la date
2. "signature" : la zone/label RÉSERVÉE à la signature (ex: "Signature du soumissionnaire", "Signataire", "Signature")
3. "cachet" : la zone/label RÉSERVÉE au cachet (ex: "Cachet du concurrent", "Cachet", "Tampon")

ATTENTION AUX FAUX POSITIFS — retourne null si :
- Le mot "signature" ou "cachet" apparaît dans le corps du texte (article, clause, phrase explicative)
  ex: "la signature du marché interviendra..." → null
  ex: "le cachet doit être apposé..." → null
- La zone n'est pas visuellement une zone vide prévue pour recevoir une signature
  (une vraie zone de signature est généralement un espace blanc avec juste le label en titre)
- La page est une page de texte courant sans zone de signature visible

Ne retourne une zone que si tu vois clairement un espace réservé à la signature/cachet
(bloc isolé, espace vide en dessous du label, cadre, ligne de signature, etc.)

Retourne EXACTEMENT ce JSON (un objet avec clé "pages", tableau de {n} entrées dans l'ordre) :
{{
  "pages": [
    {{
      "page": 1,
      "fait_a":    {{"x_pct": 0.10, "y_pct": 0.87}},
      "signature": {{"x_pct": 0.55, "y_pct": 0.92}},
      "cachet":    null
    }}
  ]
}}

Règles pour les coordonnées :
- x_pct = position horizontale du BORD GAUCHE du label (fraction de la largeur, 0.0=gauche, 1.0=droite)
- y_pct = position verticale du BAS du label (fraction de la hauteur, 0.0=haut, 1.0=bas)
  → l'image de signature sera placée JUSTE EN DESSOUS de y_pct
- null si la zone n'est pas présente sur la page
- Retourne null pour les pages sans zone particulière (corps du texte normal)"""

    # ── Construction du message avec toutes les images ─────────
    content = [{"type": "text", "text": user_text}]
    for p in pages_data:
        content.append({
            "type": "text",
            "text": f"--- Page {p['page_num']} ---"
        })
        content.append({
            "type": "image_url",
            "image_url": {
                "url":    f"data:image/png;base64,{p['b64']}",
                "detail": "high"
            }
        })

    # ── Appel API ──────────────────────────────────────────────
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": content},
        ],
        response_format={"type": "json_object"},
        max_tokens=2000,
    )

    raw = response.choices[0].message.content
    result = json.loads(raw)

    # Stats tokens
    usage = response.usage
    cost_in  = usage.prompt_tokens     * 2.50 / 1_000_000
    cost_out = usage.completion_tokens * 10.0 / 1_000_000
    print(f"Tokens input  : {usage.prompt_tokens:,}  (${cost_in:.4f})")
    print(f"Tokens output : {usage.completion_tokens:,}  (${cost_out:.4f})")
    print(f"Coût total    : ${cost_in + cost_out:.4f}")

    return result.get("pages", [])


# ── Lancement ─────────────────────────────────────────────────
print(f"Envoi de {len(pages_data)} page(s) à GPT-4o...\n")
zones_par_page = detect_zones_gpt4o(pages_data, OPENAI_API_KEY)
print()

# Affichage du résultat brut
print("Réponse GPT-4o :")
print(json.dumps(zones_par_page, indent=2, ensure_ascii=False))

## 3. Visualisation des zones détectées

Affiche uniquement les pages où GPT-4o a détecté quelque chose.

- 🟢 **Vert** = zone "Fait à"
- 🔵 **Bleu pointillé** = où la signature sera placée
- 🔴 **Rouge pointillé** = où le cachet sera placé

In [ ]:
def pct_to_rect(x_pct: float, y_pct: float, width_pt: float, height_pt: float,
                img_w: int, img_h: int, gap: int = 6) -> fitz.Rect:
    """
    Convertit des coordonnées en % (retournées par GPT-4o) en fitz.Rect de placement.

    Input  : x_pct, y_pct — position du bas du label (fractions 0-1)
             width_pt, height_pt — dimensions de la page en points PDF
             img_w, img_h — dimensions de l'image à placer
             gap — espace entre le label et l'image (points)
    Output : fitz.Rect — rectangle de destination pour insert_image()
    """
    x0 = x_pct * width_pt
    y0 = y_pct * height_pt + gap
    return fitz.Rect(x0, y0, x0 + img_w, y0 + img_h)


def page_to_image_arr(page: fitz.Page, zoom: float = 1.5) -> np.ndarray:
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    return np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)


def draw_zone(ax, x_pct, y_pct, pw, ph, zoom, img_w, img_h, color, label, gap=6):
    """Dessine le label (croix) et le rectangle de placement de l'image."""
    # Point du label (bas du texte)
    lx = x_pct * pw * zoom
    ly = y_pct * ph * zoom
    ax.plot(lx, ly, "x", color=color, markersize=10, markeredgewidth=2)
    ax.text(lx + 5, ly - 5, label, color=color, fontsize=7, fontweight="bold")
    # Rectangle de placement
    rx = x_pct * pw * zoom
    ry = (y_pct * ph + gap) * zoom
    rw = img_w * zoom
    rh = img_h * zoom
    patch = mpatches.FancyBboxPatch(
        (rx, ry), rw, rh, boxstyle="square,pad=0",
        linewidth=2, edgecolor=color, facecolor="none", linestyle="--"
    )
    ax.add_patch(patch)


doc = fitz.open(str(PDF_INPUT))

pages_with_zones = [z for z in zones_par_page
                    if z.get("fait_a") or z.get("signature") or z.get("cachet")]

if not pages_with_zones:
    print("GPT-4o n'a détecté aucune zone sur ce document.")
else:
    print(f"{len(pages_with_zones)} page(s) avec zones détectées :\n")
    for z in pages_with_zones:
        pi   = z["page"] - 1
        page = doc[pi]
        pw, ph = page.rect.width, page.rect.height
        img  = page_to_image_arr(page, ZOOM)

        fig, ax = plt.subplots(figsize=(8, 11))
        ax.imshow(img)
        ax.axis("off")

        title_parts = [f"Page {z['page']}"]

        if z.get("fait_a"):
            fa = z["fait_a"]
            ax.axhline(y=fa["y_pct"] * ph * ZOOM, color="green", linewidth=1, linestyle=":")
            ax.plot(fa["x_pct"] * pw * ZOOM, fa["y_pct"] * ph * ZOOM,
                    "x", color="green", markersize=10, markeredgewidth=2)
            ax.text(fa["x_pct"] * pw * ZOOM + 5, fa["y_pct"] * ph * ZOOM - 8,
                    "Fait à", color="green", fontsize=7, fontweight="bold")
            title_parts.append("Fait à ✓")

        if z.get("signature"):
            s = z["signature"]
            draw_zone(ax, s["x_pct"], s["y_pct"], pw, ph, ZOOM, SIG_W, SIG_H, "royalblue", "→ signature")
            title_parts.append("Signature ✓")

        if z.get("cachet"):
            c = z["cachet"]
            draw_zone(ax, c["x_pct"], c["y_pct"], pw, ph, ZOOM, CAC_W, CAC_H, "red", "→ cachet")
            title_parts.append("Cachet ✓")

        ax.set_title(" | ".join(title_parts), fontsize=11)
        plt.tight_layout()
        plt.show()

doc.close()

## 4. Application — signature + cachet + 'Fait à'

In [ ]:
def make_default_signature():
    doc = fitz.open()
    page = doc.new_page(width=200, height=58)
    page.draw_rect(fitz.Rect(3, 3, 197, 55), color=(0.42, 0.2, 0.75), width=1.5)
    page.insert_text(fitz.Point(10, 37), "Signé électroniquement", fontsize=11, color=(0.42, 0.2, 0.75))
    data = page.get_pixmap(alpha=False).tobytes("png")
    doc.close()
    return data

def make_default_cachet():
    doc = fitz.open()
    page = doc.new_page(width=110, height=110)
    page.draw_circle(fitz.Point(55, 55), 50, color=(0.42, 0.2, 0.75), width=2.5)
    page.insert_text(fitz.Point(20, 61), "CACHET", fontsize=14, color=(0.42, 0.2, 0.75))
    data = page.get_pixmap(alpha=False).tobytes("png")
    doc.close()
    return data

sig_bytes = SIGNATURE.read_bytes() if SIGNATURE.exists() else make_default_signature()
cac_bytes = CACHET.read_bytes()    if CACHET.exists()    else make_default_cachet()
print(f"Signature : {'personnalisée' if SIGNATURE.exists() else 'défaut'}")
print(f"Cachet    : {'personnalisé'  if CACHET.exists()    else 'défaut'}")
print()

# Index zones par numéro de page pour accès rapide
zones_index = {z["page"]: z for z in zones_par_page}

doc = fitz.open(str(PDF_INPUT))
n_pages = len(doc)

for i, page in enumerate(doc):
    pw, ph  = page.rect.width, page.rect.height
    page_num = i + 1
    z = zones_index.get(page_num, {})

    # ── Remplir 'Fait à' ──────────────────────────────────────
    if z.get("fait_a") and (LIEU or DATE_STR):
        fa    = z["fait_a"]
        x0_pt = fa["x_pct"] * pw
        y_pt  = fa["y_pct"] * ph
        # Efface la ligne avec un rectangle blanc
        page.draw_rect(
            fitz.Rect(x0_pt, y_pt - 14, pw - 15, y_pt + 4),
            color=(1,1,1), fill=(1,1,1)
        )
        text = "Fait \u00e0"
        if LIEU:     text += f" {LIEU}"
        if DATE_STR: text += f", le {DATE_STR}"
        page.insert_text(fitz.Point(x0_pt, y_pt), text, fontsize=10, color=(0,0,0))
        print(f"  Page {page_num} : 'Fait à' rempli ✓")

    # ── Signature + cachet à la zone détectée ─────────────────
    if z.get("signature") or z.get("cachet"):
        # Utilise la zone signature si dispo, sinon cachet
        ref = z.get("signature") or z.get("cachet")
        rx  = ref["x_pct"] * pw
        ry  = ref["y_pct"] * ph
        kw_rect = fitz.Rect(rx, ry - 12, rx + 150, ry)  # rect du label

        if z.get("cachet"):
            cr = pct_to_rect(z["cachet"]["x_pct"], z["cachet"]["y_pct"], pw, ph, CAC_W, CAC_H)
            page.insert_image(cr, stream=cac_bytes, keep_proportion=True)
            print(f"  Page {page_num} : cachet placé ✓")

        if z.get("signature"):
            sr = pct_to_rect(z["signature"]["x_pct"], z["signature"]["y_pct"], pw, ph, SIG_W, SIG_H)
            page.insert_image(sr, stream=sig_bytes, keep_proportion=True)
            print(f"  Page {page_num} : signature placée ✓")

    # ── Paraphe : signature en bas à droite de TOUTES les pages ─
    page.insert_image(
        fitz.Rect(pw - SIG_MX - SIG_W, ph - SIG_MY - SIG_H, pw - SIG_MX, ph - SIG_MY),
        stream=sig_bytes, keep_proportion=True
    )
    print(f"  Page {page_num} : paraphe bas-droite ✓")

PDF_OUTPUT.parent.mkdir(exist_ok=True)
doc.save(str(PDF_OUTPUT))
doc.close()
print(f"\n✓ PDF signé sauvegardé : {PDF_OUTPUT}")

## 5. Aperçu avant / après — pages avec zones détectées

In [ ]:
doc_in  = fitz.open(str(PDF_INPUT))
doc_out = fitz.open(str(PDF_OUTPUT))

# Montre les pages avec zones détectées + les 2 dernières pages
pages_with_zones_idx = {z["page"] - 1 for z in zones_par_page
                        if z.get("fait_a") or z.get("signature") or z.get("cachet")}
last_pages_idx = set(range(max(0, len(doc_out) - 2), len(doc_out)))
pages_to_show  = sorted(pages_with_zones_idx | last_pages_idx)

for i in pages_to_show:
    img_in  = page_to_image_arr(doc_in[i],  ZOOM)
    img_out = page_to_image_arr(doc_out[i], ZOOM)

    fig, axes = plt.subplots(1, 2, figsize=(16, 11))
    axes[0].imshow(img_in);  axes[0].axis("off"); axes[0].set_title(f"Page {i+1} — AVANT",  fontsize=11)
    axes[1].imshow(img_out); axes[1].axis("off"); axes[1].set_title(f"Page {i+1} — APRÈS", fontsize=11)
    plt.suptitle(f"Page {i+1}/{len(doc_out)}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

doc_in.close()
doc_out.close()